# Live AI Chatbot for Final Year Project
### Powered by Natural Language Processing (NLP), NLTK, TF-IDF Vectorization, and Cosine Similarity

A conversational chatbot built using Python and machine learning text processing techniques. This notebook provides an end-to-end, step-by-step walkthrough of building a retrieval-based question answering chatbot.

## Step 1: Import Necessary Libraries
We import core Python utilities, `nltk` for natural language processing, and `scikit-learn` for TF-IDF feature extraction and cosine similarity calculations.

In [1]:
import io
import random
import string
import warnings
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

## Step 2: Download NLTK Datasets
We download the tokenizers (`punkt`, `punkt_tab`) and lexical databases (`wordnet`, `omw-1.4`) required for sentence splitting and lemmatization.

In [2]:
# Downloading required NLTK packages
for package in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4']:
    nltk.download(package, quiet=True)

print('NLTK corpora and tokenizers verified.')

## Step 3: Load and Read the Knowledge Corpus
The chatbot relies on a domain knowledge corpus (`corpus.txt`). We read the text, convert it to lowercase for normalization, and tokenize it into sentences and words.

In [3]:
# Load the corpus
corpus_file = 'corpus.txt'
with open(corpus_file, 'r', encoding='utf-8', errors='ignore') as f:
    raw_data = f.read().lower()

# Sentence and word tokenization
sent_tokens = nltk.sent_tokenize(raw_data)
word_tokens = nltk.word_tokenize(raw_data)

print(f'Total Sentences in Knowledge Base: {len(sent_tokens)}')
print(f'Total Words in Knowledge Base: {len(word_tokens)}')
print('\nSample Sentence Token:')
print(sent_tokens[0])

## Step 4: Text Preprocessing (Tokenization & Lemmatization)
We define helper functions to normalize input text: removing punctuation, lowercasing, tokenizing, and reducing words to their base dictionary lemma using `WordNetLemmatizer`.

In [4]:
lemmer = WordNetLemmatizer()
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)

def LemTokens(tokens):
    return [lemmer.lemmatize(token) for token in tokens]

def LemNormalize(text):
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))

# Demonstration of Lemmatization
sample_text = 'Chatbots are playing an essential role in automating natural conversations.'
print('Original Text:', sample_text)
print('Normalized & Lemmatized Tokens:', LemNormalize(sample_text))

## Step 5: Handling Greetings and Small Talk
Before searching the knowledge base, we check if the user is greeting the bot or expressing gratitude.

In [5]:
GREETING_INPUTS = ('hello', 'hi', 'greetings', 'sup', 'what\'s up', 'hey', 'good morning')
GREETING_RESPONSES = [
    'Hi there! How can I help you today?',
    'Hello! What would you like to learn about?',
    'Hey! Feel free to ask me any question.',
    'Greetings! How may I assist you with NLP or AI topics?',
    'Hello! I am ready to answer your questions.'
]

def check_greeting(sentence):
    for word in sentence.split():
        if word.lower() in GREETING_INPUTS:
            return random.choice(GREETING_RESPONSES)
    if sentence.lower().strip() in GREETING_INPUTS:
        return random.choice(GREETING_RESPONSES)
    return None

print('Greeting Test:', check_greeting('Hello!'))

## Step 6: Generating Response via TF-IDF & Cosine Similarity
When a query is received:
1. We append the query to our corpus sentence list.
2. `TfidfVectorizer` computes the TF-IDF weight matrix.
3. `cosine_similarity` compares the query vector with each corpus sentence.
4. We retrieve the most similar sentence if the similarity score is above the threshold.

In [6]:
def generate_response(user_response, sentences):
    user_response = user_response.lower().strip()
    greeting_reply = check_greeting(user_response)
    if greeting_reply:
        return greeting_reply
        
    # Append user response to the corpus sentences
    sentences.append(user_response)
    
    try:
        tfidf_vectorizer = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english')
        tfidf_matrix = tfidf_vectorizer.fit_transform(sentences)
        
        # Calculate cosine similarity
        vals = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])
        idx = vals.argsort()[0][-1]
        flat = vals.flatten()
        flat.sort()
        req_tfidf = flat[-2]
        
        sentences.pop() # Remove user query
        
        if req_tfidf == 0 or req_tfidf < 0.15:
            return 'I am sorry, but I do not have sufficient information in my knowledge base to answer that. Could you please rephrase?'
        else:
            return sentences[idx].capitalize()
            
    except Exception as e:
        if len(sentences) > 0 and sentences[-1] == user_response:
            sentences.pop()
        return f'Error generating response: {str(e)}'

## Step 7: Testing the Chatbot
Let us test the chatbot with several queries covering greetings, domain concepts (NLP, TF-IDF, Cosine Similarity, Deep Learning), and unknown queries.

In [7]:
test_queries = [
    'Hi there',
    'What is a chatbot?',
    'What is TF-IDF?',
    'Explain cosine similarity.',
    'What is deep learning?',
    'What is the recipe for chocolate cake?'
]

print('=' * 75)
print('🤖 CHATBOT DEMO CONVERSATION')
print('=' * 75)

for query in test_queries:
    reply = generate_response(query, sent_tokens)
    print(f'User: {query}')
    print(f'Bot : {reply}\n')

## Step 8: Interactive Chatbot Loop
Run this cell to start an interactive chat session inside the notebook. Type `bye` or `quit` to exit.

In [8]:
def start_chat():
    print('Bot: Hello! I am your AI assistant. Type "bye" or "exit" to end the chat.')
    while True:
        try:
            user_input = input('You: ').strip()
            if not user_input:
                continue
            if user_input.lower() in ['bye', 'exit', 'quit']:
                print('Bot: Goodbye! Have a great day ahead.')
                break
            reply = generate_response(user_input, sent_tokens)
            print(f'Bot: {reply}\n')
        except (KeyboardInterrupt, EOFError):
            print('\nBot: Chat ended. Goodbye!')
            break

# Uncomment the following line to run interactively:
# start_chat()